# Clasificación de Pingüinos con Árbol de Decisión

Este notebook usa el dataset `penguins.csv` para entrenar un modelo de **Árbol de Decisión** que clasifica la especie de pingüino (`species`) a partir de sus características físicas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")

## 1. Carga de datos

In [ ]:
df = pd.read_csv("penguins.csv", index_col=0)
df.head()

## 2. Exploración rápida (EDA)

In [ ]:
df.info()
print()
print("Valores nulos por columna:")
print(df.isna().sum())
print()
print("Distribución de especies:")
print(df["species"].value_counts())

In [ ]:
sns.pairplot(df.dropna(), hue="species", vars=[
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"
])
plt.show()

## 3. Preprocesamiento

- Eliminamos filas con valores nulos.
- Codificamos las variables categóricas (`island`, `sex`) a valores numéricos.
- Separamos features (`X`) y target (`y = species`).

In [ ]:
df_clean = df.dropna().copy()

le_island = LabelEncoder()
le_sex = LabelEncoder()

df_clean["island"] = le_island.fit_transform(df_clean["island"])
df_clean["sex"] = le_sex.fit_transform(df_clean["sex"])

feature_cols = ["island", "bill_length_mm", "bill_depth_mm",
                 "flipper_length_mm", "body_mass_g", "sex"]

X = df_clean[feature_cols]
y = df_clean["species"]

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")

## 4. Entrenamiento del Árbol de Decisión

In [ ]:
clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)

## 5. Evaluación del modelo

In [ ]:
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print()
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
disp.plot(cmap="Blues")
plt.title("Matriz de Confusión")
plt.show()

## 6. Visualización del árbol e importancia de variables

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(
    clf,
    feature_names=feature_cols,
    class_names=clf.classes_,
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.show()

In [ ]:
importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Importancia de las variables")
plt.xlabel("Importancia")
plt.show()

importances

## 7. Guardar el modelo (.pkl)

In [ ]:
import pickle

model_bundle = {
    "model": clf,
    "feature_cols": feature_cols,
    "le_island": le_island,
    "le_sex": le_sex,
}

with open("penguins_decision_tree.pkl", "wb") as f:
    pickle.dump(model_bundle, f)

print("Modelo guardado en penguins_decision_tree.pkl")